# Regresión Logística: Fundamentos Teóricos y Optimización

## Clasificación Binaria con Entropía Cruzada y Gradiente Descendente

---

## Índice Teórico

1. **Modelo Probabilístico**
   - Función Sigmoide y su derivada
   - Interpretación como probabilidad

2. **Estimación por Máxima Verosimilitud (MLE)**
   - Función de verosimilitud
   - Log-verosimilitud
   - Derivación de la entropía cruzada

3. **Función de Costo: Entropía Cruzada Binaria**
   - Definición matemática
   - Propiedades y convexidad
   - Comparación con MSE

4. **Optimización: Gradiente Descendente**
   - Derivadas parciales
   - Actualización de parámetros
   - Learning rate y convergencia

5. **Stochastic Gradient Descent (SGD)**
   - Comparación con Batch GD
   - Mini-batch GD
   - Ventajas y desventajas

6. **Implementación de SGD desde cero**

---

## 1. Modelo Probabilístico

### 1.1 Función Sigmoide (Logística)

La función sigmoide transforma una combinación lineal en una probabilidad:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

donde $z = \mathbf{w}^T \mathbf{x} + b$

**Propiedades importantes:**
- $\sigma(z) \in (0,1)$ para todo $z \in \mathbb{R}$
- $\sigma(0) = 0.5$
- Es monótona creciente
- Es simétrica: $\sigma(-z) = 1 - \sigma(z)$

**Derivada de la sigmoide:**

$$
\frac{d\sigma(z)}{dz} = \sigma(z)(1 - \sigma(z))
$$

**Demostración:**

$$
\frac{d}{dz}\left(\frac{1}{1+e^{-z}}\right) = \frac{e^{-z}}{(1+e^{-z})^2} = \frac{1}{1+e^{-z}} \cdot \frac{e^{-z}}{1+e^{-z}} = \sigma(z)(1-\sigma(z))
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import expit
from scipy.optimize import minimize

# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

def sigmoid(z):
    """
    Función sigmoide con protección contra overflow
    σ(z) = 1 / (1 + exp(-z))
    """
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    """
    Derivada de la función sigmoide
    dσ/dz = σ(z)(1 - σ(z))
    """
    s = sigmoid(z)
    return s * (1 - s)

# Visualización
z = np.linspace(-10, 10, 1000)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sigmoide
axes[0].plot(z, sigmoid(z), linewidth=3, label='σ(z)')
axes[0].axhline(0.5, color='red', linestyle='--', alpha=0.5, label='σ(z)=0.5')
axes[0].axvline(0, color='red', linestyle='--', alpha=0.5)
axes[0].fill_between(z, 0, sigmoid(z), alpha=0.2)
axes[0].set_xlabel('z')
axes[0].set_ylabel('σ(z)')
axes[0].set_title('Función Sigmoide')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Derivada
axes[1].plot(z, sigmoid_derivative(z), linewidth=3, color='green', label='dσ/dz')
axes[1].axvline(0, color='red', linestyle='--', alpha=0.5)
axes[1].fill_between(z, 0, sigmoid_derivative(z), alpha=0.2, color='green')
axes[1].set_xlabel('z')
axes[1].set_ylabel('dσ/dz')
axes[1].set_title('Derivada de la Función Sigmoide')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Verificación numérica:")
print(f"σ(0) = {sigmoid(0):.6f}")
print(f"σ'(0) = {sigmoid_derivative(0):.6f}")
print(f"σ(10) = {sigmoid(10):.6f}")
print(f"σ(-10) = {sigmoid(-10):.6f}")

### 1.2 Modelo Probabilístico

Definimos la probabilidad de que $y=1$ dado $\mathbf{x}$:

$$
P(y=1|\mathbf{x}) = \sigma(\mathbf{w}^T\mathbf{x} + b)
$$

Y para $y=0$:

$$
P(y=0|\mathbf{x}) = 1 - \sigma(\mathbf{w}^T\mathbf{x} + b)
$$

**Notación compacta:**

$$
P(y|\mathbf{x}) = \sigma(\mathbf{w}^T\mathbf{x} + b)^y \cdot (1 - \sigma(\mathbf{w}^T\mathbf{x} + b))^{1-y}
$$

**Regla de decisión:**

$$
\hat{y} =
\begin{cases}
1 & \text{si } P(y=1|\mathbf{x}) \ge 0.5 \\
0 & \text{en otro caso}
\end{cases}
$$

**Odds y Log-Odds:**

$$
\text{Odds} = \frac{P(y=1|\mathbf{x})}{P(y=0|\mathbf{x})} = \frac{\sigma(z)}{1-\sigma(z)} = e^{z}
$$

$$
\log(\text{Odds}) = \log\left(\frac{P(y=1|\mathbf{x})}{P(y=0|\mathbf{x})}\right) = z = \mathbf{w}^T\mathbf{x} + b
$$

**Interpretación:** La regresión logística es un modelo lineal en log-odds.

## 2. Estimación por Máxima Verosimilitud (MLE)

### 2.1 Función de Verosimilitud

Dado un conjunto de datos $\mathcal{D} = \{(\mathbf{x}_i, y_i)\}_{i=1}^N$, la función de verosimilitud es:

$$
L(\mathbf{w}, b) = \prod_{i=1}^{N} P(y_i|\mathbf{x}_i; \mathbf{w}, b)
$$

Sustituyendo el modelo:

$$
L(\mathbf{w}, b) = \prod_{i=1}^{N} \sigma(\mathbf{w}^T\mathbf{x}_i + b)^{y_i} \cdot (1 - \sigma(\mathbf{w}^T\mathbf{x}_i + b))^{1-y_i}
$$

### 2.2 Log-Verosimilitud

Para facilitar la optimización, tomamos el logaritmo:

$$
\ell(\mathbf{w}, b) = \log L(\mathbf{w}, b) = \sum_{i=1}^{N} \left[ y_i \log\sigma(\mathbf{w}^T\mathbf{x}_i + b) + (1-y_i)\log(1 - \sigma(\mathbf{w}^T\mathbf{x}_i + b)) \right]
$$

### 2.3 Derivación de la Entropía Cruzada

La función de costo es el negativo de la log-verosimilitud promedio:

$$
J(\mathbf{w}, b) = -\frac{1}{N} \ell(\mathbf{w}, b)
$$

$$
J(\mathbf{w}, b) = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log\sigma(\mathbf{w}^T\mathbf{x}_i + b) + (1-y_i)\log(1 - \sigma(\mathbf{w}^T\mathbf{x}_i + b)) \right]
$$

Esta es la **Entropía Cruzada Binaria** (también llamada Log Loss).

**Interpretación desde teoría de la información:**
- Mide la diferencia entre la distribución real $y_i$ y la distribución predicha $p_i$
- La entropía cruzada $H(p,q) = -\sum p(x)\log q(x)$

**Propiedades de la Entropía Cruzada:**

1. **No negativa**: $J(\mathbf{w}, b) \ge 0$
2. **Cero sólo cuando**: $\sigma(z_i) = y_i$ para todo $i$
3. **Convexa**: tiene un único mínimo global
4. **Asimétrica**: penaliza más los errores "seguros" que los errores cerca del umbral

In [ ]:
def binary_cross_entropy(y_true, y_pred):
    """
    Entropía Cruzada Binaria
    J = -1/N Σ [y_i log(p_i) + (1-y_i) log(1-p_i)]
    """
    epsilon = 1e-15
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# Visualización de la función de costo
p = np.linspace(0.001, 0.999, 1000)

plt.figure(figsize=(10, 6))

# Costo para y=1
cost_y1 = -np.log(p)
plt.plot(p, cost_y1, label='Costo si y=1: -log(p)', linewidth=2)

# Costo para y=0
cost_y0 = -np.log(1 - p)
plt.plot(p, cost_y0, label='Costo si y=0: -log(1-p)', linewidth=2)

plt.xlabel('Probabilidad predicha p')
plt.ylabel('Costo')
plt.title('Entropía Cruzada Binaria para un solo ejemplo')
plt.axvline(0.5, color='red', linestyle='--', alpha=0.5, label='p=0.5')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0, 5)
plt.show()

# Tabla de valores
print("\nValores de costo para diferentes predicciones:")
print("-" * 50)
print("y_real | p_pred | Costo")
print("-" * 50)
test_cases = [(1, 0.99), (1, 0.5), (1, 0.01), (0, 0.01), (0, 0.5), (0, 0.99)]
for y, p in test_cases:
    cost = - (y * np.log(p) + (1-y) * np.log(1-p))
    print(f"  {y}    |  {p:.2f}    | {cost:.4f}")

## 3. Derivadas y Gradientes

### 3.1 Derivada de la Entropía Cruzada

Para un solo ejemplo $(\mathbf{x}, y)$:

$$
J_i = -\left[ y \log(\sigma(z)) + (1-y) \log(1-\sigma(z)) \right]
$$

Usando la regla de la cadena:

$$
\frac{\partial J_i}{\partial z} = -\left[ y \frac{1}{\sigma(z)} - (1-y) \frac{1}{1-\sigma(z)} \right] \cdot \frac{d\sigma(z)}{dz}
$$

$$
= -\left[ \frac{y}{\sigma(z)} - \frac{1-y}{1-\sigma(z)} \right] \cdot \sigma(z)(1-\sigma(z))
$$

$$
= -\left[ y(1-\sigma(z)) - (1-y)\sigma(z) \right] = \sigma(z) - y
$$

### 3.2 Gradientes de los Parámetros

Recordando que $z = \mathbf{w}^T\mathbf{x} + b$:

$$
\frac{\partial J_i}{\partial \mathbf{w}} = \frac{\partial J_i}{\partial z} \cdot \frac{\partial z}{\partial \mathbf{w}} = (\sigma(\mathbf{w}^T\mathbf{x} + b) - y) \cdot \mathbf{x}
$$

$$
\frac{\partial J_i}{\partial b} = \frac{\partial J_i}{\partial z} \cdot \frac{\partial z}{\partial b} = \sigma(\mathbf{w}^T\mathbf{x} + b) - y
$$

### 3.3 Gradiente del Costo Total

Para todo el dataset:

$$
\nabla_{\mathbf{w}} J(\mathbf{w}, b) = \frac{1}{N} \sum_{i=1}^{N} (\sigma(\mathbf{w}^T\mathbf{x}_i + b) - y_i) \mathbf{x}_i
$$

$$
\frac{\partial J}{\partial b} = \frac{1}{N} \sum_{i=1}^{N} (\sigma(\mathbf{w}^T\mathbf{x}_i + b) - y_i)
$$

**Interpretación de los gradientes:**
- $(\sigma(z_i) - y_i)$ es el error de predicción
- El gradiente es el error ponderado por las características
- Si la predicción es perfecta ($\sigma(z_i) = y_i$), el gradiente es cero

In [ ]:
# Verificación de gradientes con diferencias finitas
np.random.seed(42)
X_test_grad = np.random.randn(5, 3)
y_test_grad = np.random.randint(0, 2, 5)
w_test = np.random.randn(3)
b_test = np.random.randn()

def compute_gradients(X, y, w, b):
    """Calcular gradientes analíticamente"""
    z = np.dot(X, w) + b
    p = sigmoid(z)
    dw = np.dot(X.T, (p - y)) / len(y)
    db = np.mean(p - y)
    return dw, db

def numerical_gradient_w(X, y, w, b, epsilon=1e-6):
    """Calcular gradiente numérico para w"""
    grad = np.zeros_like(w)
    for i in range(len(w)):
        w_plus = w.copy()
        w_minus = w.copy()
        w_plus[i] += epsilon
        w_minus[i] -= epsilon
        
        z_plus = np.dot(X, w_plus) + b
        z_minus = np.dot(X, w_minus) + b
        
        loss_plus = binary_cross_entropy(y, sigmoid(z_plus))
        loss_minus = binary_cross_entropy(y, sigmoid(z_minus))
        
        grad[i] = (loss_plus - loss_minus) / (2 * epsilon)
    return grad

# Comparar gradientes
dw_analytical, db_analytical = compute_gradients(X_test_grad, y_test_grad, w_test, b_test)
dw_numerical = numerical_gradient_w(X_test_grad, y_test_grad, w_test, b_test)

print("Comparación de gradientes:")
print("=" * 60)
print("Índice | Analítico    | Numérico     | Diferencia")
print("-" * 60)
for i in range(len(w_test)):
    diff = abs(dw_analytical[i] - dw_numerical[i])
    print(f"  {i}    | {dw_analytical[i]:10.6f} | {dw_numerical[i]:10.6f} | {diff:.2e}")

error_relativo = np.max(np.abs(dw_analytical - dw_numerical) / (np.abs(dw_analytical) + 1e-8))
print(f"\nError relativo máximo: {error_relativo:.2e}")
print(f"¿Gradientes correctos? {'✅ Sí' if error_relativo < 1e-5 else '❌ No'}")

## 4. Optimización: Algoritmos de Gradiente Descendente

### 4.1 Batch Gradient Descent (BGD)

**Algoritmo:**
1. Inicializar $\mathbf{w}, b$
2. Para cada iteración $t$:
   - Calcular $\mathbf{g}_\mathbf{w} = \nabla_{\mathbf{w}} J(\mathbf{w}, b)$
   - Calcular $g_b = \frac{\partial J}{\partial b}$
   - Actualizar: $\mathbf{w} \leftarrow \mathbf{w} - \alpha \mathbf{g}_\mathbf{w}$
   - Actualizar: $b \leftarrow b - \alpha g_b$

**Ventajas:**
- Estimación precisa del gradiente
- Convergencia suave
- Buena para datasets pequeños

**Desventajas:**
- Costoso computacionalmente para datasets grandes
- Puede quedar atrapado en mínimos locales (convexo, no aplica aquí)

### 4.2 Stochastic Gradient Descent (SGD)

**Algoritmo:**
1. Inicializar $\mathbf{w}, b$
2. Para cada iteración $t$:
   - Barajar los datos
   - Para cada ejemplo $i$ en $1,...,N$:
     - Calcular $\mathbf{g}_\mathbf{w}^{(i)} = (\sigma(\mathbf{w}^T\mathbf{x}_i + b) - y_i) \mathbf{x}_i$
     - Calcular $g_b^{(i)} = \sigma(\mathbf{w}^T\mathbf{x}_i + b) - y_i$
     - Actualizar: $\mathbf{w} \leftarrow \mathbf{w} - \alpha \mathbf{g}_\mathbf{w}^{(i)}$
     - Actualizar: $b \leftarrow b - \alpha g_b^{(i)}$

**Ventajas:**
- Muy rápido por iteración
- Puede escapar de mínimos locales
- Funciona bien con datasets grandes
- Puede usarse en aprendizaje online

**Desventajas:**
- Gradientes ruidosos (alta varianza)
- Convergencia errática
- No aprovecha paralelización

### 4.3 Mini-Batch Gradient Descent (MBGD)

**Algoritmo:**
1. Inicializar $\mathbf{w}, b$
2. Para cada iteración $t$:
   - Barajar los datos
   - Dividir en mini-batches de tamaño $m$
   - Para cada mini-batch $\mathcal{B}$:
     - Calcular $\mathbf{g}_\mathbf{w}^{\mathcal{B}} = \frac{1}{m} \sum_{i \in \mathcal{B}} (\sigma(\mathbf{w}^T\mathbf{x}_i + b) - y_i) \mathbf{x}_i$
     - Calcular $g_b^{\mathcal{B}} = \frac{1}{m} \sum_{i \in \mathcal{B}} (\sigma(\mathbf{w}^T\mathbf{x}_i + b) - y_i)$
     - Actualizar: $\mathbf{w} \leftarrow \mathbf{w} - \alpha \mathbf{g}_\mathbf{w}^{\mathcal{B}}$
     - Actualizar: $b \leftarrow b - \alpha g_b^{\mathcal{B}}$

**Ventajas:**
- Balance entre BGD y SGD
- Puede aprovechar hardware (vectorización)
- Menos ruido que SGD
- Más rápido que BGD

**Tamaño típico de mini-batch:** 32, 64, 128, 256

## 5. Implementación de SGD desde Cero

In [ ]:
class LogisticRegressionSGD:
    """
    Regresión Logística con Stochastic Gradient Descent
    """
    def __init__(self, learning_rate=0.01, n_iterations=1000, batch_size=1, verbose=True):
        """
        Parameters:
        -----------
        learning_rate : float, tasa de aprendizaje
        n_iterations : int, número de épocas
        batch_size : int, tamaño del batch (1 para SGD, N para BGD, entre 1 y N para MBGD)
        verbose : bool, mostrar progreso
        """
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.batch_size = batch_size
        self.verbose = verbose
        self.weights = None
        self.bias = None
        self.loss_history = []
        self.gradient_norm_history = []
        
    def _initialize_parameters(self, n_features):
        """Inicialización de parámetros"""
        self.weights = np.random.randn(n_features) * 0.01
        self.bias = 0.0
        
    def _get_batches(self, X, y):
        """Genera mini-batches"""
        n_samples = X.shape[0]
        indices = np.random.permutation(n_samples)
        
        for start in range(0, n_samples, self.batch_size):
            end = min(start + self.batch_size, n_samples)
            batch_indices = indices[start:end]
            yield X[batch_indices], y[batch_indices]
            
    def _compute_gradients_batch(self, X_batch, y_batch):
        """
        Calcula los gradientes para un batch
        """
        m = X_batch.shape[0]
        z = np.dot(X_batch, self.weights) + self.bias
        p = sigmoid(z)
        
        dw = np.dot(X_batch.T, (p - y_batch)) / m
        db = np.mean(p - y_batch)
        
        return dw, db
    
    def fit(self, X, y):
        """
        Entrena el modelo con SGD
        """
        n_samples, n_features = X.shape
        self._initialize_parameters(n_features)
        
        for epoch in range(self.n_iterations):
            epoch_loss = 0
            n_batches = 0
            gradient_norm = 0
            
            # Iterar sobre los batches
            for X_batch, y_batch in self._get_batches(X, y):
                # Calcular gradientes
                dw, db = self._compute_gradients_batch(X_batch, y_batch)
                
                # Actualizar parámetros
                self.weights -= self.learning_rate * dw
                self.bias -= self.learning_rate * db
                
                # Guardar información
                gradient_norm += np.linalg.norm(dw) + abs(db)
                n_batches += 1
                
                # Calcular pérdida para este batch
                z_batch = np.dot(X_batch, self.weights) + self.bias
                p_batch = sigmoid(z_batch)
                batch_loss = binary_cross_entropy(y_batch, p_batch)
                epoch_loss += batch_loss
            
            # Promediar pérdida y norma del gradiente
            epoch_loss /= n_batches
            gradient_norm /= n_batches
            
            self.loss_history.append(epoch_loss)
            self.gradient_norm_history.append(gradient_norm)
            
            if self.verbose and (epoch % 100 == 0):
                print(f"Época {epoch:4d}, Pérdida: {epoch_loss:.6f}, ||∇||: {gradient_norm:.6f}")
        
        return self
    
    def predict_proba(self, X):
        """Predice probabilidades"""
        z = np.dot(X, self.weights) + self.bias
        return sigmoid(z)
    
    def predict(self, X, threshold=0.5):
        """Predice clases"""
        proba = self.predict_proba(X)
        return (proba >= threshold).astype(int)
    
    def score(self, X, y):
        """Calcula precisión"""
        y_pred = self.predict(X)
        return np.mean(y_pred == y)

## 6. Comparación de Algoritmos de Optimización

Generamos datos sintéticos y comparamos BGD, SGD y MBGD

In [ ]:
# Generar datos sintéticos
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = make_classification(
    n_samples=2000,
    n_features=10,
    n_informative=5,
    n_redundant=0,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Datos de entrenamiento: {X_train_scaled.shape}")
print(f"Datos de prueba: {X_test_scaled.shape}")

In [ ]:
# Entrenar diferentes versiones
configs = [
    {'name': 'Batch GD', 'batch_size': len(X_train_scaled)},
    {'name': 'Mini-Batch GD (64)', 'batch_size': 64},
    {'name': 'Mini-Batch GD (128)', 'batch_size': 128},
    {'name': 'SGD', 'batch_size': 1},
]

models = []
plt.figure(figsize=(15, 10))

for idx, config in enumerate(configs):
    # Entrenar modelo
    model = LogisticRegressionSGD(
        learning_rate=0.01,
        n_iterations=500,
        batch_size=config['batch_size'],
        verbose=False
    )
    model.fit(X_train_scaled, y_train)
    models.append(model)
    
    # Evaluar
    train_acc = model.score(X_train_scaled, y_train)
    test_acc = model.score(X_test_scaled, y_test)
    
    # Graficar pérdida
    plt.subplot(2, 2, idx+1)
    plt.plot(model.loss_history, label='Pérdida')
    plt.plot(model.gradient_norm_history, label='Norma del Gradiente', alpha=0.7)
    plt.xlabel('Época')
    plt.ylabel('Valor')
    plt.title(f"{config['name']}\nTrain Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    
plt.tight_layout()
plt.show()

# Comparación de tiempos
print("\n" + "="*60)
print("Comparación de Algoritmos de Optimización")
print("="*60)
print(f"{'Algoritmo':20s} | {'Batch Size':10s} | {'Train Acc':10s} | {'Test Acc':10s}")
print("-"*60)
for config, model in zip(configs, models):
    train_acc = model.score(X_train_scaled, y_train)
    test_acc = model.score(X_test_scaled, y_test)
    print(f"{config['name']:20s} | {config['batch_size']:10d} | {train_acc:9.4f} | {test_acc:9.4f}")

## 7. Análisis Teórico: Convexidad y Convergencia

### 7.1 Convexidad de la Entropía Cruzada

La función de costo $J(\mathbf{w}, b)$ es convexa.

**Matriz Hessiana:**

$$
\mathbf{H} = \nabla^2 J(\mathbf{w}, b) = \frac{1}{N} \sum_{i=1}^{N} \sigma(\mathbf{w}^T\mathbf{x}_i + b)(1 - \sigma(\mathbf{w}^T\mathbf{x}_i + b)) \mathbf{x}_i \mathbf{x}_i^T
$$

La Hessiana es semidefinida positiva porque $\sigma(z)(1-\sigma(z)) > 0$ para todo $z$.

### 7.2 Tasa de Convergencia

Para BGD con learning rate $\alpha$:

$$
J(\mathbf{w}_{t+1}) - J(\mathbf{w}^*) \le \frac{\|\mathbf{w}_0 - \mathbf{w}^*\|^2}{2\alpha t}
$$

Para SGD:

$$
\mathbb{E}[J(\mathbf{w}_t) - J(\mathbf{w}^*)] \le O\left(\frac{1}{\sqrt{t}}\right)
$$

### 7.3 Learning Rate Scheduling

Para mejorar la convergencia de SGD:

**Decaimiento por tiempo:**

$$
\alpha_t = \frac{\alpha_0}{1 + \beta t}
$$

**Decaimiento exponencial:**

$$
\alpha_t = \alpha_0 \cdot \gamma^t
$$

**Decaimiento por pasos:**

$$
\alpha_t = \alpha_0 \cdot \gamma^{\lfloor t / k \rfloor}
$$

In [ ]:
# Experimentos con diferentes learning rates
learning_rates = [0.0001, 0.001, 0.01, 0.05, 0.1, 0.5]

plt.figure(figsize=(15, 10))

results = []
for i, lr in enumerate(learning_rates):
    model = LogisticRegressionSGD(
        learning_rate=lr,
        n_iterations=300,
        batch_size=64,
        verbose=False
    )
    model.fit(X_train_scaled, y_train)
    results.append((lr, model.loss_history, model.score(X_test_scaled, y_test)))
    
    plt.subplot(2, 3, i+1)
    plt.plot(model.loss_history, label=f'LR={lr}')
    plt.xlabel('Época')
    plt.ylabel('Pérdida')
    plt.title(f'Learning Rate = {lr}\nTest Acc: {model.score(X_test_scaled, y_test):.4f}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.ylim(0, 2)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("Efecto del Learning Rate")
print("="*60)
print(f"{'Learning Rate':15s} | {'Test Accuracy':15s} | {'Iteraciones hasta convergencia'}")
print("-"*70)
for lr, loss_hist, test_acc in results:
    # Encontrar cuándo converge (cambio < 1e-4)
    converged = len(loss_hist)
    for t in range(10, len(loss_hist)):
        if abs(loss_hist[t] - loss_hist[t-1]) < 1e-4:
            converged = t
            break
    print(f"{lr:15.4f} | {test_acc:15.4f} | {converged:15d} épocas")

## 8. Comparación con Scikit-Learn y Optimizadores

Comparación con diferentes solvers implementados en scikit-learn

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import time

# Solver de scikit-learn
solvers = ['lbfgs', 'liblinear', 'sag', 'saga']
sklearn_results = []

print("\n" + "="*70)
print("Comparación con Scikit-Learn")
print("="*70)
print(f"{'Solver':12s} | {'Tiempo':10s} | {'Train Acc':10s} | {'Test Acc':10s} | {'Coef Norm':10s}")
print("-"*70)

for solver in solvers:
    start = time.time()
    model = LogisticRegression(solver=solver, max_iter=1000, random_state=42)
    model.fit(X_train_scaled, y_train)
    elapsed = time.time() - start
    
    train_acc = accuracy_score(y_train, model.predict(X_train_scaled))
    test_acc = accuracy_score(y_test, model.predict(X_test_scaled))
    coef_norm = np.linalg.norm(model.coef_)
    
    sklearn_results.append((solver, elapsed, train_acc, test_acc, coef_norm))
    print(f"{solver:12s} | {elapsed:9.3f}s | {train_acc:9.4f} | {test_acc:9.4f} | {coef_norm:9.4f}")

# Modelo SGD implementado
model_sgd = LogisticRegressionSGD(
    learning_rate=0.01,
    n_iterations=1000,
    batch_size=64,
    verbose=False
)

start = time.time()
model_sgd.fit(X_train_scaled, y_train)
elapsed = time.time() - start

train_acc = model_sgd.score(X_train_scaled, y_train)
test_acc = model_sgd.score(X_test_scaled, y_test)
coef_norm = np.linalg.norm(model_sgd.weights)

print(f"{'SGD (ours)':12s} | {elapsed:9.3f}s | {train_acc:9.4f} | {test_acc:9.4f} | {coef_norm:9.4f}")

## 9. Resumen Teórico

### 9.1 Ecuaciones Fundamentales

**Modelo:**

$$
P(y=1|\mathbf{x}) = \sigma(\mathbf{w}^T\mathbf{x} + b), \quad \sigma(z) = \frac{1}{1+e^{-z}}
$$

**Función de Costo (Entropía Cruzada):**

$$
J(\mathbf{w}, b) = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log\sigma(\mathbf{w}^T\mathbf{x}_i + b) + (1-y_i)\log(1 - \sigma(\mathbf{w}^T\mathbf{x}_i + b)) \right]
$$

**Gradientes:**

$$
\nabla_{\mathbf{w}} J = \frac{1}{N} \sum_{i=1}^{N} (\sigma(\mathbf{w}^T\mathbf{x}_i + b) - y_i) \mathbf{x}_i
$$

$$
\frac{\partial J}{\partial b} = \frac{1}{N} \sum_{i=1}^{N} (\sigma(\mathbf{w}^T\mathbf{x}_i + b) - y_i)
$$

**Actualización (SGD):**

$$
\mathbf{w} \leftarrow \mathbf{w} - \alpha (\sigma(\mathbf{w}^T\mathbf{x}_i + b) - y_i) \mathbf{x}_i
$$

$$
b \leftarrow b - \alpha (\sigma(\mathbf{w}^T\mathbf{x}_i + b) - y_i)
$$

In [ ]:
print("\n" + "="*70)
print("RESUMEN: COMPARACIÓN DE ALGORITMOS DE OPTIMIZACIÓN")
print("="*70)

print("""
┌──────────────────────────┬──────────────┬──────────────┬──────────────┐
│      Característica      │  Batch GD    │   SGD        │ Mini-Batch GD│
├──────────────────────────┼──────────────┼──────────────┼──────────────┤
│ Uso de datos             │ Todo dataset │ 1 ejemplo    │ m ejemplos   │
│ Precisión del gradiente  │ Exacto       │ Ruidoso      │ Balanceado   │
│ Velocidad por iteración  │ Lento        │ Rápido       │ Rápido       │
│ Convergencia             │ Suave        │ Errática     │ Moderada     │
│ Atascado en mínimos      │ Sí           │ No           │ A veces      │
│ Paralelizable            │ Sí           │ No           │ Sí           │
│ Uso memoria              │ Alto         │ Bajo         │ Medio        │
│ Recomendado para         │ Datos pequeños│ Datos grandes│ Ambos        │
└──────────────────────────┴──────────────┴──────────────┴──────────────┘
""")

print("\n📊 Comparación de precisión final:")
print("-"*40)
for config, model in zip(configs, models):
    test_acc = model.score(X_test_scaled, y_test)
    print(f"{config['name']:20s} : {test_acc:.4f}")

print(f"{'Scikit-Learn (lbfgs)':20s} : {sklearn_results[0][3]:.4f}")

## 10. Ejercicios Teóricos

### Ejercicio 1: Derivación

Demuestra que:

$$
\frac{\partial}{\partial w_j} \left[-\log P(y|\mathbf{x})\right] = (\sigma(\mathbf{w}^T\mathbf{x} + b) - y) x_j
$$

### Ejercicio 2: Propiedades de la Entropía Cruzada

1. Muestra que $J(\mathbf{w}, b) \ge 0$
2. Encuentra el mínimo de $J_i$ para un ejemplo individual

### Ejercicio 3: Hessiana

Calcula la matriz Hessiana de $J(\mathbf{w}, b)$ y demuestra que es semidefinida positiva.

### Ejercicio 4: SGD con Momentum

Modifica la actualización de SGD para incluir momentum:

$$
v_{t+1} = \beta v_t + (1-\beta) \nabla J(\mathbf{w}_t)
$$

$$
\mathbf{w}_{t+1} = \mathbf{w}_t - \alpha v_{t+1}
$$

### Ejercicio 5: Regularización

Añade regularización L2 a la función de costo:

$$
J_{reg}(\mathbf{w}, b) = J(\mathbf{w}, b) + \frac{\lambda}{2} \|\mathbf{w}\|^2
$$

Deriva los nuevos gradientes.

## 11. Referencias

1. **Bishop, C. M. (2006)**. *Pattern Recognition and Machine Learning*. Springer.
2. **Hastie, T., Tibshirani, R., & Friedman, J. (2009)**. *The Elements of Statistical Learning*. Springer.
3. **Goodfellow, I., Bengio, Y., & Courville, A. (2016)**. *Deep Learning*. MIT Press.
4. **Murphy, K. P. (2012)**. *Machine Learning: A Probabilistic Perspective*. MIT Press.

---

**Fin del Notebook Teórico** 📚